In [1]:
import sys
import asyncio
import time
import os

import numpy as np

from lsst.ts import salobj

from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from lsst.ts.observatory.control.utils import RotType

/opt/lsst/software/stack/conda/miniconda3-py38_4.9.2/envs/lsst-scipipe-0.4.1/lib/python3.8/site-packages/jose/backends/cryptography_backend.py:23: CryptographyDeprecationWarning: int_from_bytes is deprecated, use int.from_bytes instead
  from cryptography.utils import int_from_bytes, int_to_bytes


In [3]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [2]:
!eups list -s lsst_distrib

   21.0.0-1-g00ce914+aad3ba2940 	current w_2021_10 setup


In [4]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [5]:
#Start classes
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.02 sec
Read 1 history items for RemoteEvent(ATArchiver, 0, authList)
Read 1 history items for RemoteEvent(ATArchiver, 0, errorCode)
Read 100 history items for RemoteEvent(ATArchiver, 0, heartbeat)
Read 100 history items for RemoteEvent(ATArchiver, 0, imageInOODS)
Read 100 history items for RemoteEvent(ATArchiver, 0, imageRetrievalForArchiving)
Read 1 history items for RemoteEvent(ATArchiver, 0, logLevel)
Read 1 history items for RemoteEvent(ATArchiver, 0, logMessage)
Read 1 history items for RemoteEvent(ATArchiver, 0, simulationMode)
Read historical data in 0.07 sec
Read 1 history items for RemoteE

[[None, None, None, None, None, None, None], [None, None, None, None]]

trajectory DDS read queue is filling: 37 of 100 elements
torqueDemand DDS read queue is filling: 37 of 100 elements
mount_positions DDS read queue is filling: 38 of 100 elements
nasymth_m3_mountMotorEncoders DDS read queue is filling: 37 of 100 elements
mountStatus DDS read queue is full (100 elements); data may be lost
mount_Nasmyth_Encoders DDS read queue is filling: 37 of 100 elements
guidingAndOffsets DDS read queue is full (100 elements); data may be lost
mount_AzEl_Encoders DDS read queue is filling: 37 of 100 elements
currentTargetStatus DDS read queue is full (100 elements); data may be lost
mount_AzEl_Encoders python read queue is filling: 36 of 100 elements
measuredTorque DDS read queue is filling: 37 of 100 elements
measuredMotorVelocity DDS read queue is filling: 37 of 100 elements
azEl_mountMotorEncoders DDS read queue is filling: 37 of 100 elements


In [ ]:
# enable components
#await atcs.enable()
#await latiss.enable()

In [ ]:
await atcs.prepare_for_flatfield()

In [ ]:
await latiss.take_bias(1)

In [ ]:
# Take 50 biases seq # 2-51
# Added wait to stop killing the recent images
for i in range(50):
    await asyncio.sleep(2.0)
    await latiss.take_bias(1)

In [ ]:
# Take 10 10 second darks 52-61
await latiss.take_darks(10.0, 10)

In [ ]:
# Take 10 2 second flats 62-71
await latiss.take_flats(2.0, 10, filter='RG610', grating='empty_1')

In [ ]:
# Take flats for PTC 72-111
# Added wait to stop killing the recent images
for i in range(20):
    exp = 0.2 * float(i+1)
    await latiss.take_flats(exp, 1, filter='empty_1', grating='empty_1')
    if exp < 2.0:
        await asyncio.sleep(2.0)
    await latiss.take_flats(exp, 1, filter='empty_1', grating='empty_1')


In [ ]:
await latiss.take_bias(1)

In [ ]:
await atcs.stop_tracking()

In [ ]:
atcs.atdome = True
atcs.atdometrajectory = True

In [ ]:
await atcs.slew_dome_to(az=0.0)

In [ ]:
await atcs.prepare_for_onsky()

In [ ]:
await salobj.set_summary_state(atcs.rem.ataos, salobj.State.STANDBY)

In [ ]:
# Don't forget the 'settingToApply' or it won't work!!
await salobj.set_summary_state(atcs.rem.ataos, salobj.State.ENABLED, settingsToApply='current')

In [ ]:
await atcs.rem.ataos.cmd_enableCorrection.set_start(
    m1=True, hexapod=True, atspectrograph=True, timeout=atcs.long_timeout
)

In [ ]:
# HD72514 Az~120, Alt~65 m=8.05

In [ ]:
current_az = 120.0
current_el = 65.0
current_rot = -50.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

In [ ]:
await salobj.set_summary_state(atcs.rem.atptg, salobj.State.ENABLED)

In [ ]:
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.ENABLED)

In [6]:
# HD55070 m = 5.45, Az = +85, Alt = +74
# HD78876 m = 7.2, Az = +90, Alt = +60
# HD72514 m = 
await atcs.slew_object('HD72514', rot=-50.0, rot_type=RotType.PhysicalSky)

Slewing to HD72514: 08 31 56.9423 -39 03 58.547
Setting rotator physical position to -50.0 deg. Rotator will track sky.
Parallactic angle: -59.57442927013727 | Sky Angle: -4.0555782942404335
xml 7 compatibility mode: rotPA=-4.0555782942404335, rotFrame=1
Sending command
Stop tracking.
target python read queue is filling: 20 of 100 elements
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
atdome: <State.ENABLED: 2>
atdometrajectory: <State.ENABLED: 2>
[Telescope] delta Alt = +000.013 deg; delta Az = +000.016 deg; delta N1 = +000.000 deg; delta N2 = +006.131 deg [Dome] delta Az = -002.558 deg
ATDome in position.
[Telescope] delta Alt = -000.002 deg; delta Az = -000.004 deg; delta N1 = +000.000 deg; delta N2 = +002.681 deg [Dome] delta Az = -002.558 deg
[Telescope] delta Alt =

In [ ]:
#await latiss.take_object(10.0, 1, filter='empty_1', grating='empty_1')

In [ ]:
#await latiss.setup_atspec(filter='RG610')

In [ ]:
#await latiss.take_object(10.0, 1, filter='empty_1', grating='empty_1')

In [ ]:
#await latiss.setup_atspec

In [ ]:
#await latiss.setup_instrument

In [7]:
# Ran this command after re-starting notebook, but it was too much.  
# The cell had put the offset back to the starting point (which was -0.20),
# So I only should have applied -0.09.  So image 137 was way out of focus.
await atcs.rem.ataos.cmd_offset.set_start(z=-0.29)

In [8]:
await latiss.take_object(10.0, 1, filter='RG610', grating='empty_1')

Generating group_id
OBJECT 0001 - 0001


array([2021031000137])

In [ ]:
# await latiss.take_object(10.0, 1, filter='empty_1', grating='empty_1')

In [ ]:
# Take a set of images on both sides of current best focus

starting_z_offset = -0.20
z_offset_increment = 0.05
nsteps = int((-2 * starting_z_offset) / z_offset_increment) + 1
total_z_offset = 0.0
await atcs.rem.ataos.cmd_offset.set_start(z=starting_z_offset)
total_z_offset += starting_z_offset
print(f"Total z offset = {total_z_offset}")
await asyncio.sleep(2)
for i in range(nsteps):
    await latiss.take_object(10.0, 1, filter='RG610', grating='empty_1')
    await atcs.rem.ataos.cmd_offset.set_start(z=z_offset_increment)
    total_z_offset += z_offset_increment
    print(f"Total z offset = {total_z_offset}")
    
# Put offset back where it was
await atcs.rem.ataos.cmd_offset.set_start(z=-total_z_offset)
total_z_offset -= total_z_offset
print(f"Total z offset = {total_z_offset}")
    

In [ ]:
# To reset all offsets
tmp = await atcs.rem.ataos.cmd_resetOffset.set_start(axis='z')

In [ ]:
await latiss.take_object(1.0, 1, filter='empty_1', grating='empty_1')

In [9]:
await atcs.shutdown()

Disabling ATAOS corrections
Disabling ATAOS corrections.
Closing M1 cover vent gates.
Cover state <MirrorCoverState.OPENED: 7>
Closing M1 cover.
Cover state <MirrorCoverState.OPENED: 7>
Cover state <MirrorCoverState.INMOTION: 8>
Cover state <MirrorCoverState.CLOSED: 6>
M1 vent state <VentsPosition.OPENED: 0>
Closing M1 vents.
M1 vent state <VentsPosition.PARTIALLYOPENED: 2>
M1 vent state <VentsPosition.CLOSED: 1>
Close dome.
Closing dome shutter...
process as completed...
atdome: <State.ENABLED: 2>
Waiting for ATDome mainDoorState: <ShutterDoorState.CLOSED: 1>. Current state: <ShutterDoorState.OPENED: 2>.
mainDoorState: <ShutterDoorState.CLOSING: 5>
mainDoorState: <ShutterDoorState.CLOSED: 1>
Finishing ATDome shutter command task.
ATDome shutter command task not done. Cancelling.
ATDome shutter command task cancelled.
Slew dome to Park position.
Sending ATDomeTrajectory to DISABED state. Component will be left in DISABLEDstate or else it may send the ATDome back to alignment with the t

RuntimeError: Failed to transition ['atdome'] to <State.STANDBY: 5>.

In [ ]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply="test")

In [11]:
# Putting everything back in standby.
await latiss.standby()

[atcamera]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atspectrograph]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atheaderservice]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atarchiver]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
All components in <State.STANDBY: 5>.
